# Lab 03 · Pandas for Data Manipulation and Cleaning
**CS-3105 Machine Learning Lab** · Lab Instructor: Engr. Muhammad Awais

**CLO 6** (C-4 Analyze, PLO 2 Problem Analysis): *Analyze real-world datasets by applying data manipulation and visualization techniques.*

**Data:** `pakistan_used_cars.csv`, 24,973 real used-car ads from OLX Pakistan (Open Data Pakistan, *Pakistan Used Cars*, 2021). The ads are real and contain the problems real data has. Two kinds of formatting problem were added for practice (some prices written as text such as `Rs 1,430,000`, some city names in the wrong case or with extra spaces).

**Rules:** no Python loops over rows · label every output, e.g. `print("T4e:", ...)` · do not edit the CSV by hand · Runtime → Restart and run all before you submit · push this notebook to `ML-Lab-<RollNo>/Lab03/Lab03.ipynb` by **Thursday 5 November 2026, 11:59 PM (PKT)**.

**Name:** Muhammad Bilal Khan          **Roll No:** 2024-CET-20

## Warm-up (not graded): the class example from the lecture
Run this cell and repeat any lecture example on this small table before you start the tasks. `clean` is the tidy version, `messy` has the six kinds of problem.

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.width', 140)

clean = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Zainab"],
    "City": ["Muzaffarabad", "Mirpur", "Bagh", "Muzaffarabad", "Rawalakot", "Mirpur", "Kotli", "Muzaffarabad"],
    "Maths": [78, 91, 45, 85, 62, 73, 38, 88],
    "Physics": [65, 88, 52, 79, 48, 81, 41, 92],
    "Attendance": [92, 97, 70, 88, 75, 90, 60, 95]})

messy = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-04", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Fatima", "Zainab"],
    "City": [" Muzaffarabad", "Mirpur", "Bagh", "MUZAFFARABAD", "Rawalakot", "Mirpur", "Kotli", "MUZAFFARABAD", "Muzafarabad"],
    "Maths": [78, 91, np.nan, 85, 62, 73, 380, 85, 88],
    "Physics": [65, 88, 52, 79, 48, 105, 41, 79, 92],
    "Attendance": ["92%", "97", "70", "88", "75%", "90", "60", "88", "95"]})
print(clean)

    Roll    Name          City  Maths  Physics  Attendance
0  SE-01     Ali  Muzaffarabad     78       65          92
1  SE-02  Ayesha        Mirpur     91       88          97
2  SE-03   Bilal          Bagh     45       52          70
3  SE-04  Fatima  Muzaffarabad     85       79          88
4  SE-05   Hamza     Rawalakot     62       48          75
5  SE-06    Sana        Mirpur     73       81          90
6  SE-07   Usman         Kotli     38       41          60
7  SE-08  Zainab  Muzaffarabad     88       92          95


In [2]:
# Warm-up: lecture examples on the small class table

print("W1: students with Maths > 80")
print(clean[clean["Maths"] > 80])

print("")
print("W2: Muzaffarabad AND Maths > 80")
print(clean[(clean["City"] == "Muzaffarabad") & (clean["Maths"] > 80)])

print("")
print("W3: top 3 by Physics")
print(clean.nlargest(3, "Physics"))

print("")
print("W4: messy Maths mean (wrong because of 380 and the duplicate) =", messy["Maths"].mean())
print("W5: messy duplicates =", messy.duplicated().sum())

W1: students with Maths > 80
    Roll    Name          City  Maths  Physics  Attendance
1  SE-02  Ayesha        Mirpur     91       88          97
3  SE-04  Fatima  Muzaffarabad     85       79          88
7  SE-08  Zainab  Muzaffarabad     88       92          95

W2: Muzaffarabad AND Maths > 80
    Roll    Name          City  Maths  Physics  Attendance
3  SE-04  Fatima  Muzaffarabad     85       79          88
7  SE-08  Zainab  Muzaffarabad     88       92          95

W3: top 3 by Physics
    Roll    Name          City  Maths  Physics  Attendance
7  SE-08  Zainab  Muzaffarabad     88       92          95
1  SE-02  Ayesha        Mirpur     91       88          97
5  SE-06    Sana        Mirpur     73       81          90

W4: messy Maths mean (wrong because of 380 and the duplicate) = 117.75
W5: messy duplicates = 1


### Setup
Put `pakistan_used_cars.csv` in the same folder as this notebook. In Colab: click the folder icon on the left → Upload → choose the CSV.

## Task 1 · Load and first look
- **(a)** Read `pakistan_used_cars.csv` into `df` and print its shape.
- **(b)** Show the first 5 rows.
- **(c)** Print the column names.
- **(d)** Show the data type of every column.
- **(e)** Which column should be a number but is not? Show two of its values that explain why.

In [3]:
# T1: load and first look

df = pd.read_csv("pakistan_used_cars.csv")
print("T1a: shape =", df.shape)                      
print("T1b: first 5 rows")                            
print(df.head(5))
print("T1c: column names =", list(df.columns))       
print("T1d: data types")                           
print(df.dtypes)

rs_example = df.loc[df["Price"].str.contains("Rs"), "Price"].iloc[0]
print("T1e: column = Price | example 1:", df["Price"].iloc[0], "| example 2:", rs_example)

T1a: shape = (24973, 9)
T1b: first 5 rows
    Brand Condition    Fuel  KMs Driven        Model    Price Registered City Transaction Type    Year
0  Toyota      Used  Diesel         1.0        Prado  2100000         Karachi             Cash  1997.0
1  Suzuki      Used  Petrol    100000.0        Bolan   380000         Karachi             Cash  2006.0
2  Suzuki      Used     CNG     12345.0        Bolan   340000         Karachi             Cash  1998.0
3  Suzuki      Used  Petrol     94000.0         Alto   535000         Karachi             Cash  2010.0
4  Toyota      Used  Petrol    100000.0  Corolla XLI  1430000         Karachi             Cash  2013.0
T1c: column names = ['Brand', 'Condition', 'Fuel', 'KMs Driven', 'Model', 'Price', 'Registered City', 'Transaction Type', 'Year']
T1d: data types
Brand                   str
Condition               str
Fuel                    str
KMs Driven          float64
Model                   str
Price                   str
Registered City         st

**T1e answer:** `Price` should be a number but is stored as text (`str`/object). Most values look like `2100000`, but some are written as `Rs 435,000`; the `"Rs"` and the commas force the whole column to be text.

## Task 2 · Retrieve data
- **(a)** The `Brand` column (first 5 values).
- **(b)** The columns `Brand`, `Model`, `Price` together (first 5 rows).
- **(c)** The first row, using `iloc`.
- **(d)** The `Model` in the row with index 100, using `loc`.
- **(e)** How many cars are registered in `"Islamabad"`?
- **(f)** How many **Honda** cars run on **Hybrid** fuel?
- **(g)** Cars from 2018 or newer, newest first (Brand, Model, Year, Price; first 10).
- **(h)** Try `df[df["Price"] > 1_000_000]`. What happens and why? Answer in a Markdown cell.

In [4]:
# T2: retrieve data (on the raw, uncleaned df)

print("T2a: Brand (first 5)")
print(df["Brand"].head(5))
print("T2b: Brand, Model, Price (first 5 rows)")
print(df[["Brand", "Model", "Price"]].head(5))
print("T2c: first row with iloc")
print(df.iloc[0])
print("T2d: Model in row with index 100 =", df.loc[100, "Model"])
print("T2e: cars registered in Islamabad =", (df["Registered City"] == "Islamabad").sum())
print("T2f: Honda cars on Hybrid fuel =", ((df["Brand"] == "Honda") & (df["Fuel"] == "Hybrid")).sum())
print("T2g: cars from 2018 or newer, newest first (top 10)")
print(df[df["Year"] >= 2018].sort_values("Year", ascending=False)[["Brand", "Model", "Year", "Price"]].head(10))

try:
    df[df["Price"] > 1_000_000]
except TypeError as e:
    print("T2h: TypeError ->", e)

T2a: Brand (first 5)
0    Toyota
1    Suzuki
2    Suzuki
3    Suzuki
4    Toyota
Name: Brand, dtype: str
T2b: Brand, Model, Price (first 5 rows)
    Brand        Model    Price
0  Toyota        Prado  2100000
1  Suzuki        Bolan   380000
2  Suzuki        Bolan   340000
3  Suzuki         Alto   535000
4  Toyota  Corolla XLI  1430000
T2c: first row with iloc
Brand                Toyota
Condition              Used
Fuel                 Diesel
KMs Driven              1.0
Model                 Prado
Price               2100000
Registered City     Karachi
Transaction Type       Cash
Year                 1997.0
Name: 0, dtype: object
T2d: Model in row with index 100 = Mehran VX
T2e: cars registered in Islamabad = 958
T2f: Honda cars on Hybrid fuel = 202
T2g: cars from 2018 or newer, newest first (top 10)
          Brand                      Model    Year         Price
10940    Toyota                     Probox  2020.0       1200000
13795     Honda                        Fit  2020.0       19

**T2h answer:** `df[df["Price"] > 1_000_000]` raises a `TypeError` (`'>' not supported between instances of 'str' and 'int'`). `Price` is still a *text* column because some values are written like `"Rs 435,000"`, and text cannot be compared with a number. Once `Price` is cleaned and converted to `int64` (Task 3), the same filter works.

## Task 3 · Clean the data: the six types of problem
Fix the problems **in this order** (text first, then duplicates):
- **(a) Missing values:** count them per column. Drop ads with no `Brand` or no `Model`; fill missing `Registered City`, `Fuel` and `Transaction Type` with `"Unknown"`; leave `Year` and `KMs Driven` as NaN. Print how many rows were removed.
- **(b) Formatting:** remove extra spaces and fix the case of city names (`.str.strip().str.title()`); remove `"Rs"` and commas from `Price`.
- **(c) Data types:** convert `Price` to `int64` and `Year` to `Int64`.
- **(d) Inconsistent categories:** `"Sukkar"` → `"Sukkur"` and `"Corrolla Altis"` → `"Corolla Altis"`. Print the number of different cities before and after.
- **(e) Duplicates:** count exact duplicate ads, drop them, print the new row count.
- **(f) Invalid values:** set `KMs Driven` to NaN when it is below 100, above 1,000,000, or the placeholder 12345 / 123456. Drop ads with the fake prices 12,345,678 or 87,654,321, and Suzuki/Daihatsu ads above 5,000,000.
- **(g) Self-check:** your final shape must be **(20484, 9)**.

In [5]:
# T3: clean the data, in order 1 -> 6

print("T3a: missing values per column")
print(df.isna().sum())
rows_before = len(df)
df = df.dropna(subset=["Brand", "Model"])                      
for col in ["Registered City", "Fuel", "Transaction Type"]:     
    df[col] = df[col].fillna("Unknown")

print("T3a: rows removed =", rows_before - len(df), "| rows left =", len(df))

df["Registered City"] = df["Registered City"].str.strip().str.title()
df["Price"] = (df["Price"].str.replace("Rs", "", regex=False)
                          .str.replace(",", "", regex=False)
                          .str.strip())
print("T3b: Price after removing Rs and commas (still text):", df["Price"].head(3).tolist())

df["Price"] = df["Price"].astype("int64")
df["Year"] = df["Year"].astype("Int64")
print("T3c: data types after conversion")
print(df.dtypes)

print("T3d: different cities BEFORE =", df["Registered City"].nunique())
df["Registered City"] = df["Registered City"].replace({"Sukkar": "Sukkur"})
df["Model"] = df["Model"].replace({"Corrolla Altis": "Corolla Altis"})
print("T3d: different cities AFTER  =", df["Registered City"].nunique())

print("T3e: exact duplicate ads =", df.duplicated().sum())
df = df.drop_duplicates()
print("T3e: row count after dropping duplicates =", len(df))

bad_km = ((df["KMs Driven"] < 100) | (df["KMs Driven"] > 1_000_000)
          | df["KMs Driven"].isin([12345, 123456]))
df.loc[bad_km, "KMs Driven"] = np.nan
print("T3f: KMs Driven set to NaN =", bad_km.sum())
bad_price = (df["Price"].isin([12_345_678, 87_654_321])
             | (df["Brand"].isin(["Suzuki", "Daihatsu"]) & (df["Price"] > 5_000_000)))
print("T3f: ads dropped for fake prices =", bad_price.sum())
df = df[~bad_price]

print("T3g: final shape =", df.shape, "| expected (20484, 9) ->", df.shape == (20484, 9))

T3a: missing values per column
Brand               2137
Condition           2136
Fuel                2445
KMs Driven          2286
Model               2448
Price                  0
Registered City     4636
Transaction Type    2445
Year                2284
dtype: int64
T3a: rows removed = 2448 | rows left = 22525
T3b: Price after removing Rs and commas (still text): ['2100000', '380000', '340000']
T3c: data types after conversion
Brand                   str
Condition               str
Fuel                    str
KMs Driven          float64
Model                   str
Price                 int64
Registered City         str
Transaction Type        str
Year                  Int64
dtype: object
T3d: different cities BEFORE = 62
T3d: different cities AFTER  = 61
T3e: exact duplicate ads = 2028
T3e: row count after dropping duplicates = 20497
T3f: KMs Driven set to NaN = 2852
T3f: ads dropped for fake prices = 13
T3g: final shape = (20484, 9) | expected (20484, 9) -> True


## Task 4 · Analyse
- **(a)** New column `Price_lakh` = price in lakh rupees (÷ 100,000).
- **(b)** New column `Age` = 2020 − `Year`.
- **(c)** The 5 most common brands.
- **(d)** The 5 cities with the most ads.
- **(e)** Median `Price_lakh` for each fuel type, highest first.
- **(f)** Median `Price_lakh` of a **Corolla GLI** for each year 2010–2018. How much value does it lose per year?
- **(g)** The 5 most expensive ads. Does one of them look wrong? Why?
- **(h)** Number of ads and median price for Cash vs Installment/Leasing. Interpret in 1–2 lines.

In [6]:
# T4: analyse

df["Price_lakh"] = df["Price"] / 100_000                                  
print("T4a: Price vs Price_lakh")
print(df[["Price", "Price_lakh"]].head())
df["Age"] = 2020 - df["Year"]                                            
print("T4b: Year vs Age")
print(df[["Year", "Age"]].head())
print("T4c: 5 most common brands")                                         
print(df["Brand"].value_counts().head(5))
print("T4d: 5 cities with the most ads")                                   
print(df["Registered City"].value_counts().head(5))
print("T4e: median Price_lakh per fuel type, highest first")               
print(df.groupby("Fuel")["Price_lakh"].median().sort_values(ascending=False))

gli = df[(df["Model"] == "Corolla GLI") & df["Year"].between(2010, 2018)]
gli_median = gli.groupby("Year")["Price_lakh"].median()
print("T4f: median Price_lakh of Corolla GLI by year")
print(gli_median)
print("T4f: change from the previous year (lakh)")
print(gli_median.diff())
loss = (gli_median.iloc[-1] - gli_median.iloc[0]) / (len(gli_median) - 1)
print("T4f: average value lost per year of age (lakh) =", round(loss, 2))

print("T4g: 5 most expensive ads")                                        
print(df.nlargest(5, "Price")[["Brand", "Model", "Year", "KMs Driven", "Condition", "Price_lakh"]])

print("T4h: Cash vs Installment/Leasing")                                 
print(df.groupby("Transaction Type")["Price_lakh"].agg(["count", "median"]).loc[["Cash", "Installment/Leasing"]])

T4a: Price vs Price_lakh
     Price  Price_lakh
0  2100000       21.00
1   380000        3.80
2   340000        3.40
3   535000        5.35
4  1430000       14.30
T4b: Year vs Age
   Year  Age
0  1997   23
1  2006   14
2  1998   22
3  2010   10
4  2013    7
T4c: 5 most common brands
Brand
Suzuki      8621
Toyota      4535
Honda       3025
Daihatsu    2231
Nissan       694
Name: count, dtype: int64
T4d: 5 cities with the most ads
Registered City
Karachi       15461
Unknown        2040
Lahore         1601
Islamabad       777
Rawalpindi      119
Name: count, dtype: int64
T4e: median Price_lakh per fuel type, highest first
Fuel
Hybrid    16.2750
Diesel    13.0000
Petrol     8.7525
CNG        3.8500
LPG        1.5750
Name: Price_lakh, dtype: float64
T4f: median Price_lakh of Corolla GLI by year
Year
2010    12.250
2011    13.325
2012    14.125
2013    14.950
2014    16.750
2015    17.900
2016    19.000
2017    21.000
2018    23.200
Name: Price_lakh, dtype: float64
T4f: change from the previ

**T4f answer:** The median Corolla GLI price rises steadily with the model year, from about **12.25 lakh (2010)** to **23.20 lakh (2018)**. Going back in age, a Corolla GLI loses about **1.4 lakh rupees per year** on average ((23.20 − 12.25) / 8 ≈ 1.37 lakh), roughly half its value over 8 years. The yearly drop is not constant: it is smaller for older cars (about 0.8 lakh per year around 2012–2013) and larger for the newest ones (about 2 lakh per year in 2017–2018), so cars lose value fastest when they are young.

**T4g answer:** Yes. The most expensive ad is a **2017 Toyota Land Cruiser at 670 lakh (Rs 67 million)**, listed as "New" but with 32,222 km driven, which is contradictory (a new car has not driven 32,000 km), and the repeating digits look typed in. It is also far above the other Land Cruisers from 2015 or later (median about 225 lakh) and 150 lakh above the second-place ad. Other top entries are suspicious too (e.g. a 2007 Toyota Surf marked "New" at 410 lakh). It is an *outlier*, not an impossible value, so I flag it and would check the ad rather than delete it automatically.

**T4h answer:** Cash ads dominate (**19,634 ads, median 6.9 lakh**), while Installment/Leasing has only **850 ads with a median of 3.0 lakh**. Installment/leasing ads are mostly cheaper cars bought by people who cannot pay everything at once. Part of the gap may also be that some installment ads show only the down payment instead of the full price, so the two medians are not fully comparable.

## Task 5 · Save your work
Save the cleaned DataFrame as `pakistan_used_cars_clean.csv` (Lab 04 uses it) and print its shape.

In [7]:
# T5: save the cleaned data

df.to_csv("pakistan_used_cars_clean.csv", index=False)
print("T5: saved pakistan_used_cars_clean.csv | shape =", df.shape)

T5: saved pakistan_used_cars_clean.csv | shape = (20484, 11)


## Task 6 · Reflection (Markdown cell)
Write 3–5 sentences: which of the six cleaning problems would have given the most misleading answer if you had ignored it, and two things you learned about the used-car market in Pakistan.

**Reflection.** The most misleading problem would have been **invalid values**: about 2,850 kilometre readings (like 1 km or the placeholder 12345) and 13 fake prices (like Rs 87,654,321 for a Suzuki) raise no error and would silently distort every average and price-vs-age result. Duplicates (2,028 repeated ads, about 8%) would also have inflated counts, and a text `Price` column would have stopped the analysis completely. About the market: it is dominated by **Suzuki (about 42% of ads)** and by **Karachi (about 75% of ads)**, and **hybrid cars have the highest median price (about 16.3 lakh)**, far above petrol (8.8 lakh) and CNG (3.9 lakh). Also, used cars lose value quickly: a Corolla GLI loses about 1.4 lakh per year.

## Bonus Exercises ##

In [8]:
# Bonus 1: brand with the highest median price (only brands with at least 50 ads)
print("")
print("Bonus 1: brand with the highest median price (only brands with at least 50 ads)")
print("")

counts = df["Brand"].value_counts()
big = counts[counts >= 50].index
b1 = df[df["Brand"].isin(big)].groupby("Brand")["Price_lakh"].median().sort_values(ascending=False)
print("B1: median Price_lakh by brand (brands with >= 50 ads), top 5")
print(b1.head(5))
print("B1: highest median brand =", b1.index[0], "->", b1.iloc[0], "lakh")

# Bonus 2: flag prices more than 5x the median price of the same 
print("")
print("Bonus 2: flag prices more than 5x the median price of the same")
print("")

model_median = df.groupby("Model")["Price"].transform("median")
flag = df["Price"] > 5 * model_median
print("B2: ads priced more than 5x their model's median =", flag.sum())
print(df[flag].nlargest(5, "Price")[["Brand", "Model", "Year", "Condition", "Price_lakh"]])

# Bonus 3: are "New" cars really new? New ads with Year before 2015
print("")
print("Bonus 3: are 'New' cars really new? New ads with Year before 2015")
print("")

old_new = (df["Condition"] == "New") & (df["Year"] < 2015)
print("B3: 'New' ads with Year before 2015 =", old_new.sum(), "out of", (df["Condition"] == "New").sum(), "New ads")


Bonus 1: brand with the highest median price (only brands with at least 50 ads)

B1: median Price_lakh by brand (brands with >= 50 ads), top 5
Brand
Mercedes    15.750
Toyota      13.200
Honda        9.000
Mazda        8.600
Nissan       7.625
Name: Price_lakh, dtype: float64
B1: highest median brand = Mercedes -> 15.75 lakh

Bonus 2: flag prices more than 5x the median price of the same

B2: ads priced more than 5x their model's median = 97
        Brand         Model  Year Condition  Price_lakh
19688  Toyota  Land Cruiser  2017       New       670.0
22246  Toyota  Land Cruiser  2005      Used       520.0
19716  Toyota  Land Cruiser  2015       New       420.0
22323  Toyota          Surf  2007       New       410.0
22688  Toyota         Prius  2015       New       357.5

Bonus 3: are 'New' cars really new? New ads with Year before 2015

B3: 'New' ads with Year before 2015 = 2035 out of 3889 New ads


**Bonus answers:**
- **B1:** Among brands with at least 50 ads, **Mercedes** has the highest median price (**15.75 lakh**), followed by Toyota (13.2 lakh) and Honda (9.0 lakh).
- **B2:** **97 ads** are priced more than 5 times the median of their own model. Many are the Land Cruiser and Surf ads seen in T4g. They are only *flagged* for checking, not deleted.
- **B3:** **No, not all "New" cars are new.** 2,035 of the 3,889 "New" ads (about 52%) have a Year before 2015, so "New" in these ads often just means unused or unregistered, not recently made.